In [1]:
"""
train_tender_model_m2.py
========================
Навчання M2-моделі. Структура та сама що в train_tender_model.py,
але читає features_tenders_full_m2.parquet і зберігає у artifacts/m2/.
"""

import json
import warnings
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import optuna
import lightgbm as lgb

from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    f1_score, precision_score, recall_score,
    precision_recall_curve, roc_curve,
    classification_report, confusion_matrix,
)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ══════════════════════════════════════════════════════════════
# НАЛАШТУВАННЯ M2
# ══════════════════════════════════════════════════════════════

MODEL_NAME       = "m2"
FEATURES_PATH    = "features_tenders_full_m2.parquet"
DASU_LABELS_PATH = "dasu_labels.parquet"
ARTIFACTS_DIR    = Path("artifacts") / MODEL_NAME
ARTIFACTS_DIR.mkdir(exist_ok=True, parents=True)

RANDOM_STATE     = 42
N_OPTUNA_TRIALS  = 40
SAMPLE_FOR_HPO   = 1_000_000

TRAIN_CUTOFF     = pd.Timestamp("2024-07-01")
VAL_CUTOFF       = pd.Timestamp("2025-01-01")

NON_FEATURE_COLS = [
    "tender_id", "buyer_id", "tender_start_date",
    "dasu_label", "main_cpv_2_digit",
]

RISK_THRESHOLDS = {
    "Низький":    (0.00, 0.05),
    "Помірний":   (0.05, 0.15),
    "Високий":    (0.15, 0.30),
    "Критичний":  (0.30, 1.01),
}


# ══════════════════════════════════════════════════════════════
# 1. ЗАВАНТАЖЕННЯ
# ══════════════════════════════════════════════════════════════

print("═" * 60)
print(f"  НАВЧАННЯ {MODEL_NAME.upper()}-МОДЕЛІ")
print("═" * 60)

df = pd.read_parquet(FEATURES_PATH)
df["tender_start_date"] = pd.to_datetime(df["tender_start_date"])
print(f"Тендерів: {len(df):,}")

vc = df["dasu_label"].value_counts()
print(f"label=0: {vc.get(0,0):,} ({vc.get(0,0)/len(df):.2%})")
print(f"label=1: {vc.get(1,0):,} ({vc.get(1,0)/len(df):.2%})")

dasu_all = pd.read_parquet(DASU_LABELS_PATH, columns=["tender_id"])
certified_ids = set(dasu_all["tender_id"].astype(str).str.strip().tolist())
print(f"Certified (з реальним моніторингом): {len(certified_ids):,}")


# ══════════════════════════════════════════════════════════════
# 2. ROZBIVKA
# ══════════════════════════════════════════════════════════════

feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
print(f"\nОзнак: {len(feature_cols)}")
print(f"  {feature_cols}")

mask_train = df["tender_start_date"] < TRAIN_CUTOFF
mask_val   = (df["tender_start_date"] >= TRAIN_CUTOFF) & (df["tender_start_date"] < VAL_CUTOFF)
mask_test  = df["tender_start_date"] >= VAL_CUTOFF
mask_test_certified = mask_test & df["tender_id"].isin(certified_ids)

X_train = df.loc[mask_train, feature_cols].copy()
y_train = df.loc[mask_train, "dasu_label"].copy()

X_val   = df.loc[mask_val,   feature_cols].copy()
y_val   = df.loc[mask_val,   "dasu_label"].copy()

X_test_full = df.loc[mask_test,           feature_cols].copy()
y_test_full = df.loc[mask_test,           "dasu_label"].copy()
X_test_cert = df.loc[mask_test_certified, feature_cols].copy()
y_test_cert = df.loc[mask_test_certified, "dasu_label"].copy()

medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(medians)
X_val   = X_val.fillna(medians)
X_test_full = X_test_full.fillna(medians)
X_test_cert = X_test_cert.fillna(medians)


# ══════════════════════════════════════════════════════════════
# 3. OPTUNA HPO
# ══════════════════════════════════════════════════════════════

spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"\nscale_pos_weight: {spw:.1f}")
print(f"Optuna: {N_OPTUNA_TRIALS} спроб на підвибірці {SAMPLE_FOR_HPO:,}")

def stratified_sample(X, y, n):
    if len(X) <= n:
        return X, y
    pos_idx = y[y == 1].index
    neg_idx = y[y == 0].index
    n_pos = min(len(pos_idx), int(n * y.mean()) + 1)
    n_neg = n - n_pos
    samp = (
        pd.Index(np.random.RandomState(RANDOM_STATE).choice(neg_idx, n_neg, replace=False))
        .append(pd.Index(np.random.RandomState(RANDOM_STATE).choice(pos_idx, n_pos, replace=False)))
    )
    return X.loc[samp], y.loc[samp]

X_hpo, y_hpo = stratified_sample(X_train, y_train, SAMPLE_FOR_HPO)


def objective(trial):
    params = {
        "objective": "binary", "metric": "average_precision",
        "verbosity": -1, "boosting_type": "gbdt",
        "random_state": RANDOM_STATE, "scale_pos_weight": spw,
        "n_jobs": -1,
        "num_leaves":        trial.suggest_int("num_leaves", 31, 127),
        "max_depth":         trial.suggest_int("max_depth", 4, 10),
        "learning_rate":     trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
        "n_estimators":      trial.suggest_int("n_estimators", 300, 800),
        "min_child_samples": trial.suggest_int("min_child_samples", 50, 500),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
    }
    m = lgb.LGBMClassifier(**params)
    m.fit(X_hpo, y_hpo, eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
    return average_precision_score(y_val, m.predict_proba(X_val)[:, 1])


study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print("BEST PARAMS:", study.best_params)
print("BEST VALUE:", study.best_value)

best_params = {
    **study.best_params,
    "objective": "binary", "metric": "average_precision",
    "verbosity": -1, "boosting_type": "gbdt",
    "random_state": RANDOM_STATE, "scale_pos_weight": spw, "n_jobs": -1,
}
print(f"\nНайкращий val PR-AUC: {study.best_value:.4f}")


# ══════════════════════════════════════════════════════════════
# 4. ФІНАЛЬНЕ НАВЧАННЯ
# ══════════════════════════════════════════════════════════════

n_calib = int(len(X_train) * 0.20)
X_sub, X_calib = X_train.iloc[:-n_calib], X_train.iloc[-n_calib:]
y_sub, y_calib = y_train.iloc[:-n_calib], y_train.iloc[-n_calib:]

lgb_model = lgb.LGBMClassifier(**best_params)
lgb_model.fit(X_sub, y_sub, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(100)])

val_raw = lgb_model.predict_proba(X_val)[:, 1]
print(f"\n[Val] PR-AUC (без калібрування): {average_precision_score(y_val, val_raw):.4f}")
print(f"[Val] ROC-AUC:                   {roc_auc_score(y_val, val_raw):.4f}")


# ══════════════════════════════════════════════════════════════
# 5. КАЛІБРУВАННЯ
# ══════════════════════════════════════════════════════════════

calib_raw = lgb_model.predict_proba(X_calib)[:, 1]
calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(calib_raw, y_calib)

val_cal       = calibrator.predict(val_raw)
test_cert_raw = lgb_model.predict_proba(X_test_cert)[:, 1]
test_cert_cal = calibrator.predict(test_cert_raw)
test_full_raw = lgb_model.predict_proba(X_test_full)[:, 1]
test_full_cal = calibrator.predict(test_full_raw)

print(f"[Val] PR-AUC (калібрований): {average_precision_score(y_val, val_cal):.4f}")


# ══════════════════════════════════════════════════════════════
# 6. ПОРІГ + МЕТРИКИ
# ══════════════════════════════════════════════════════════════

prec_v, rec_v, thr_v = precision_recall_curve(y_val, val_cal)
f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-9)
best_i = np.argmax(f1_v)
best_threshold = float(thr_v[best_i])
print(f"\nОптимальний поріг: {best_threshold:.4f}")


def report(y_true, y_prob, name):
    pred = (y_prob >= best_threshold).astype(int)
    print(f"\n── {name} ──")
    print(f"  Розмір:      {len(y_true):,}")
    print(f"  Позитивних:  {y_true.sum():,} ({y_true.mean():.2%})")
    print(f"  PR-AUC:      {average_precision_score(y_true, y_prob):.4f}")
    print(f"  ROC-AUC:     {roc_auc_score(y_true, y_prob):.4f}")
    if y_true.sum() > 0:
        print(f"  F1:          {f1_score(y_true, pred):.4f}")
        print(f"  Precision:   {precision_score(y_true, pred):.4f}")
        print(f"  Recall:      {recall_score(y_true, pred):.4f}")


report(y_test_cert, test_cert_cal, "TEST CERTIFIED")
report(y_test_full, test_full_cal, "TEST FULL")


# ══════════════════════════════════════════════════════════════
# 7. SHAP
# ══════════════════════════════════════════════════════════════

explainer = shap.TreeExplainer(lgb_model)
sample = X_test_cert.sample(min(2000, len(X_test_cert)), random_state=RANDOM_STATE)
sv = explainer.shap_values(sample)
sv = sv[1] if isinstance(sv, list) else sv

importance = (
    pd.DataFrame({"feature": feature_cols,
                  "mean_abs_shap": np.abs(sv).mean(axis=0)})
    .sort_values("mean_abs_shap", ascending=False)
)
print(f"\nТоп-10 ознак ({MODEL_NAME}):")
print(importance.head(10).to_string(index=False))


# ══════════════════════════════════════════════════════════════
# 8. ЗБЕРЕЖЕННЯ
# ══════════════════════════════════════════════════════════════

joblib.dump(lgb_model,   ARTIFACTS_DIR / "model.pkl")
joblib.dump(calibrator,  ARTIFACTS_DIR / "calibrator.pkl")
joblib.dump(explainer,   ARTIFACTS_DIR / "explainer.pkl")
joblib.dump(medians,     ARTIFACTS_DIR / "medians.pkl")
importance.to_csv(ARTIFACTS_DIR / "shap_importance.csv", index=False)

meta = {
    "model_name":       MODEL_NAME,
    "best_threshold":   best_threshold,
    "feature_cols":     feature_cols,
    "risk_thresholds":  RISK_THRESHOLDS,
    "scale_pos_weight": float(spw),
    "best_params":      study.best_params,
    "test_certified_metrics": {
        "pr_auc":  float(average_precision_score(y_test_cert, test_cert_cal)),
        "roc_auc": float(roc_auc_score(y_test_cert, test_cert_cal)),
    },
}
with open(ARTIFACTS_DIR / "meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"\n✅ Артефакти збережено у {ARTIFACTS_DIR}/")

C:\Users\legen\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


════════════════════════════════════════════════════════════
  НАВЧАННЯ M2-МОДЕЛІ
════════════════════════════════════════════════════════════
Тендерів: 922,575
label=0: 908,147 (98.44%)
label=1: 14,428 (1.56%)
Certified (з реальним моніторингом): 78,132

Ознак: 20
  ['proc_method_enc', 'non_price_criteria', 'is_works', 'is_services', 'is_high_risk_cpv', 'log_tender_value', 'submission_window_days', 'has_submission_window', 'number_of_items', 'number_of_documents', 'is_buyer_masked', 'is_weekend', 'is_q4', 'is_december', 'has_enquiries', 'buyer_violation_rate', 'buyer_total_tenders', 'value_vs_cpv_median', 'window_vs_cpv_median', 'value_vs_buyer_median']

scale_pos_weight: 45.6
Optuna: 40 спроб на підвибірці 1,000,000


Best trial: 11. Best value: 0.281157: 100%|██████████| 40/40 [05:18<00:00,  7.95s/it]


BEST PARAMS: {'num_leaves': 127, 'max_depth': 8, 'learning_rate': 0.032983003609067585, 'n_estimators': 782, 'min_child_samples': 291, 'subsample': 0.8810796931031994, 'colsample_bytree': 0.9840665681373126, 'reg_alpha': 0.13310835139010238, 'reg_lambda': 0.3678035987240854}
BEST VALUE: 0.2811568578858386

Найкращий val PR-AUC: 0.2812
[100]	valid_0's average_precision: 0.260386
[200]	valid_0's average_precision: 0.269553

[Val] PR-AUC (без калібрування): 0.2701
[Val] ROC-AUC:                   0.9629
[Val] PR-AUC (калібрований): 0.2574

Оптимальний поріг: 0.1508

── TEST CERTIFIED ──
  Розмір:      4,962
  Позитивних:  2,998 (60.42%)
  PR-AUC:      0.7773
  ROC-AUC:     0.7306
  F1:          0.5701
  Precision:   0.7868
  Recall:      0.4470

── TEST FULL ──
  Розмір:      292,319
  Позитивних:  2,998 (1.03%)
  PR-AUC:      0.2779
  ROC-AUC:     0.9657
  F1:          0.3423
  Precision:   0.2774
  Recall:      0.4470

Топ-10 ознак (m2):
               feature  mean_abs_shap
  buyer_vio